In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
x = pd.read_csv('../data/raw/TCGA-PANCAN-HiSeq-801x20531/data.csv', index_col=0)

y = pd.read_csv('../data/raw/TCGA-PANCAN-HiSeq-801x20531/labels.csv', index_col=0)
y = y.squeeze()

x.head()

,gene_0,gene_1,gene_2,gene_3,gene_4,gene_5,gene_6,gene_7,gene_8,gene_9,...,gene_20521,gene_20522,gene_20523,gene_20524,gene_20525,gene_20526,gene_20527,gene_20528,gene_20529,gene_20530
sample_0,0.0,2.017209,3.265527,5.478487,10.431999,0.0,7.175175,0.591871,0.0,0.0,...,4.926711,8.210257,9.723516,7.220030,9.119813,12.003135,9.650743,8.921326,5.286759,0.0
sample_1,0.0,0.592732,1.588421,7.586157,9.623011,0.0,6.816049,0.000000,0.0,0.0,...,4.593372,7.323865,9.740931,6.256586,8.381612,12.674552,10.517059,9.397854,2.094168,0.0
sample_2,0.0,3.511759,4.327199,6.881787,9.870730,0.0,6.972130,0.452595,0.0,0.0,...,5.125213,8.127123,10.908640,5.401607,9.911597,9.045255,9.788359,10.090470,1.683023,0.0
sample_3,0.0,3.663618,4.507649,6.659068,10.196184,0.0,7.843375,0.434882,0.0,0.0,...,6.076566,8.792959,10.141520,8.942805,9.601208,11.392682,9.694814,9.684365,3.292001,0.0
sample_4,0.0,2.655741,2.821547,6.539454,9.738265,0.0,6.566967,0.360982,0.0,0.0,...,5.996032,8.891425,10.373790,7.181162,9.846910,11.922439,9.217749,9.461191,5.110372,0.0


In [3]:
y.head()

sample_0    PRAD
sample_1    LUAD
sample_2    PRAD
sample_3    PRAD
sample_4    BRCA
Name: Class, dtype: object

Investigating if the data is skewed and requires a log(x+1). Z=(x-μ)/σ is most representative when the data is near symmetric. With skewed data a few extreme values will inflate the mean / variance causing most samples to cluster near one z-score after scaling reducing the ability of the model to differentiate and classify correctly.

In [4]:
from scipy.stats import skew

In [5]:
skew_raw = x.apply(skew)
print(f"raw skew: {skew_raw.median():.3f}")

raw skew: 0.146


In [6]:
log_x = np.log1p(x)
transformed_skew = log_x.apply(skew)
print(f"transformed skew: {transformed_skew.median():.3f}")

transformed skew: -0.314


The original right skew is actually quite small well below 1 - applying the transformation makes the values more left skewed so the log transformation will not be applied.

In [7]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=67, stratify=y)
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

(640, 20531)
(640,)
(161, 20531)
(161,)


In [8]:
print("training class:")
print(y_train.value_counts(normalize=True).sort_index())
print()
print("testing class:")
print(y_test.value_counts(normalize=True).sort_index())
print()
print("full dataset:")
print(y.value_counts(normalize=True).sort_index())

training class:
Class
BRCA    0.375000
COAD    0.096875
KIRC    0.181250
LUAD    0.176563
PRAD    0.170313
Name: proportion, dtype: float64

testing class:
Class
BRCA    0.372671
COAD    0.099379
KIRC    0.186335
LUAD    0.173913
PRAD    0.167702
Name: proportion, dtype: float64

full dataset:
Class
BRCA    0.374532
COAD    0.097378
KIRC    0.182272
LUAD    0.176030
PRAD    0.169788
Name: proportion, dtype: float64


Stratified sampling being used due to the same problem mentioned in the EDA. Class samples vary slightly, but this is negligible.

In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(x_train)
x_train = pd.DataFrame(scaler.transform(x_train), columns=x_train.columns, index=x_train.index)
x_test = pd.DataFrame(scaler.transform(x_test), columns=x_test.columns, index=x_test.index)

print(f"training mean: {x_train.mean().mean()}, training std: {x_train.std().mean()}")
print(f"test mean: {x_test.mean().mean()}, test std: {x_test.std().mean()}")

training mean: -5.193017366762975e-19, training std: 0.987328565921545
test mean: -0.005169222971301403, test std: 0.9737656185359934


Test set mean is -0.005 (not exactly 0) and std is 0.974 (not exactly 1). 
This is expected because the scaler was fitted on training data only — 
the test set is scaled using training statistics, which is correct behaviour 
for preventing data leakage.

## Label Encoding

Converting string cancer type labels to integers for scikit-learn compatibility. 
LabelEncoder assigns integers alphabetically. Fitted on the full target since 
label encoding does not use any feature information (no leakage risk).

In [10]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(y)

y_train = le.transform(y_train)
y_test = le.transform(y_test)

print(dict(zip(le.classes_, le.transform(le.classes_))))

{'BRCA': np.int64(0), 'COAD': np.int64(1), 'KIRC': np.int64(2), 'LUAD': np.int64(3), 'PRAD': np.int64(4)}


## Saving Processed Data

Saving scaled features, encoded labels, scaler, and label encoder so 
downstream notebooks can load them directly without re-running this notebook.

In [11]:
import os
import joblib

os.makedirs('../data/processed', exist_ok=True)

x_train.to_csv('../data/processed/x_train.csv')
x_test.to_csv('../data/processed/x_test.csv')
pd.Series(y_train, name='label').to_csv('../data/processed/y_train.csv', index=False)
pd.Series(y_test, name='label').to_csv('../data/processed/y_test.csv', index=False)

joblib.dump(scaler, '../data/processed/scaler.joblib')
joblib.dump(le, '../data/processed/label_encoder.joblib')

print('Saved to ../data/processed/')

Saved to ../data/processed/


## Phase 2 Summary

- **Log1p**: Skipped — raw median skewness 0.146 (near-symmetric). 
log1p would over-correct to -0.314.
- **Train/test split**: 640 / 161 (80/20), stratified. 
Class proportions verified consistent.
- **StandardScaler**: Fitted on training data only. 
Train mean ≈ 0, test mean ≈ -0.005 (no data leakage).
- **Label encoding**: BRCA=0, COAD=1, KIRC=2, LUAD=3, PRAD=4
- **Saved**: Processed data and sklearn objects to `data/processed/`

**Next**: Phase 3 — Feature Selection